# Reproducible health-claim verification

This notebook runs genuine inference only. It validates the benchmark, prevents few-shot leakage, logs raw responses, and implements a true two-stage translation condition.

In [ ]:
!pip -q install "transformers>=4.46,<5" "accelerate>=1.1,<2" "huggingface_hub>=0.25,<1" "pandas>=2.2,<3" "numpy<3" "scipy<2" "scikit-learn<2" "statsmodels>=0.14,<1" sentencepiece

import json, os, re, time, uuid
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import torch
from google.colab import files, userdata


In [ ]:
# Upload final_benchmark_70.csv
uploaded = files.upload()
benchmark_path = next(iter(uploaded))
df = pd.read_csv(benchmark_path)
required = {"claim_id","topic","claim_type","english","urdu","roman_urdu","gold_label","source_url"}
assert required <= set(df.columns), required - set(df.columns)
assert len(df) == 70 and df.claim_id.nunique() == 70
assert df.gold_label.value_counts().to_dict() == {"Supported":24,"Refuted":23,"Insufficient":23}
print("Validated 70-claim benchmark")


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_REVISION = None  # optional exact commit SHA
SEED = 42
MAX_NEW_TOKENS = 64

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, token=HF_TOKEN, device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16 if torch.cuda.is_available() else torch.float32
)
model.eval()
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
LABELS = ["Supported", "Refuted", "Insufficient"]
def normalize_label(text):
    text = " ".join(str(text or "").strip().lower().split())
    matches = [label for label in LABELS if label.lower() in text and len(text) <= 80]
    return matches[0] if len(matches) == 1 else None

def generate(prompt):
    messages = [{"role":"user","content":prompt}]
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) if hasattr(tokenizer,"apply_chat_template") else prompt
    inputs = tokenizer(rendered, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output = model.generate(**inputs, do_sample=False, max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def verify_prompt(claim):
    return f"Classify the claim as exactly one of Supported, Refuted, or Insufficient. Return only the label.\nClaim: {claim}\nLabel:"

def translate_prompt(claim, form):
    return f"Translate this {form} health claim into English. Preserve negation, modality, quantifiers, numbers, units, dates, medical terms, and qualifiers. Return only the translation.\nClaim: {claim}\nEnglish translation:"


In [ ]:
# Optional separate few-shot file. Leave None to skip few-shot.
demos = None
# demos = pd.read_csv("fewshot_demonstrations.csv")
# assert not (set(demos.claim_id) & set(df.claim_id)), "Few-shot leakage"


In [ ]:
def run(condition, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as handle:
        for item in df.itertuples():
            for form in ("english","urdu","roman_urdu"):
                claim = getattr(item, form)
                translation_prompt = None
                translation_raw = None
                if condition == "translate_then_verify" and form != "english":
                    translation_prompt = translate_prompt(claim, form)
                    translation_raw = generate(translation_prompt)
                    verification_claim = translation_raw
                else:
                    verification_claim = claim
                verification_prompt = verify_prompt(verification_claim)
                started = time.time()
                error = None
                try:
                    raw = generate(verification_prompt)
                except Exception as exc:
                    raw = ""
                    error = repr(exc)
                record = {
                    "run_id": str(uuid.uuid4()),
                    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                    "model_requested": MODEL_ID,
                    "model_revision_requested": MODEL_REVISION,
                    "prompting_condition": condition,
                    "claim_id": item.claim_id, "topic": item.topic, "claim_type": item.claim_type,
                    "form": form, "claim_text": claim, "gold_label": item.gold_label,
                    "translation_prompt": translation_prompt, "translation_raw": translation_raw,
                    "verification_prompt": verification_prompt, "raw_response": raw,
                    "predicted_label": normalize_label(raw),
                    "invalid_reason": "unparseable_or_empty" if normalize_label(raw) is None else None,
                    "seed": SEED, "temperature": 0.0, "max_new_tokens": MAX_NEW_TOKENS,
                    "latency_seconds": time.time()-started, "error": error
                }
                handle.write(json.dumps(record, ensure_ascii=False)+"\n")
    print("Saved", output_path)

# Run one condition at a time; uncomment after reviewing configuration.
# run("zero_shot", "results/zero_shot.jsonl")
# run("translate_then_verify", "results/two_stage_translate_then_verify.jsonl")


## Analysis

After raw files are saved, run the repository pipeline in Colab or locally:

`python -m src.analysis --raw results/two_stage_translate_then_verify.jsonl --out results/analysis`

The pipeline reports all-example accuracy, valid-output accuracy, invalid rates, macro-F1, paired tests, and claim-cluster bootstrap intervals.